### Ретросинтез 500 молекул с помощью AiZynthFinder

Расчет занял ~6.4 часов.

In [ ]:
import logging
import json
import time
from pathlib import Path

import pandas as pd
from rdkit import Chem

from aizynthfinder.aizynthfinder import AiZynthFinder
from aizynthfinder.utils.logging import setup_logger


# ------------------------------
# ПУТИ
# ------------------------------
BASE_DIR = Path(".").resolve()

SDF_INPUT = BASE_DIR / "output_500.sdf"
SMILES_TXT = BASE_DIR / "output_500.smiles"
CONFIG_YAML = BASE_DIR / "aizynth_data" / "config.yml"

ROUTES_JSONL = BASE_DIR / "routes_500.jsonl"
RESULTS_CSV = BASE_DIR / "results_500.csv"


# ------------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ------------------------------
def sdf_to_smiles(sdf_path: Path, smiles_path: Path):
    suppl = Chem.SDMolSupplier(str(sdf_path))
    smiles_list = []
    for mol in suppl:
        if mol is None:
            continue
        smi = Chem.MolToSmiles(mol)
        smiles_list.append(smi)
    with open(smiles_path, "w") as f:
        for smi in smiles_list:
            f.write(smi + "\n")


def run_retrosynthesis_and_save():
    """
    1) Прогоняет молекулы через AiZynthFinder.
    2) Сохраняет:
       - routes_500.jsonl: все маршруты (деревья) по каждой молекуле,
       - results_500.csv: статистику поиска по каждой молекуле,
         включая best_state_score.
    """
    setup_logger(console_level=logging.INFO)

    finder = AiZynthFinder(str(CONFIG_YAML))

    # выбираем политики и stock один раз
    finder.expansion_policy.select("uspto")
    finder.filter_policy.select("uspto")
    finder.stock.select("zinc")

    smiles = [line.strip() for line in open(SMILES_TXT) if line.strip()]

    rows = []
    t0 = time.time()

    with open(ROUTES_JSONL, "w") as f_routes:
        for idx, smi in enumerate(smiles, start=1):
            print(f"[{idx}/{len(smiles)}] Target: {smi}")
            finder.target_smiles = smi

            # ретросинтетический поиск
            finder.tree_search()
            finder.build_routes()

            routes = finder.routes  # RouteCollection

            # ------- 1. сохраняем все маршруты в JSONL -------
            route_dicts = routes.dicts
            enriched_routes = []
            for rid, rdict in enumerate(route_dicts):
                enriched_routes.append({
                    "route_id": rid,
                    "tree": rdict,
                })

            record = {
                "target": smi,
                "routes": enriched_routes,
            }
            f_routes.write(json.dumps(record) + "\n")

            # ------- 2. статистика поиска -------
            stats = finder.extract_statistics()

            # нормализуем статус: если пустой, выводим solved/unsolved по n_routes
            raw_status = stats.get("status")
            if raw_status:
                status = raw_status
            else:
                status = "unsolved" if len(route_dicts) == 0 else "solved"

            # ------- 3. best_state_score по найденным маршрутам -------
            best_state_score = None
            if route_dicts:
                routes.compute_scores(*finder.scorers.objects())
                all_scores = routes.all_scores  # список dict'ов по маршрутам

                state_values = []
                for s in all_scores:
                    if "state score" in s:
                        v = s["state score"]
                    elif "state_score" in s:
                        v = s["state_score"]
                    else:
                        v = None
                    if v is not None:
                        state_values.append(v)

                if state_values:
                    best_state_score = max(state_values)

            rows.append({
                "target": smi,
                "n_routes": len(route_dicts),
                "search_time": stats.get("search_time"),
                "first_solution_time": stats.get("first_solution_time"),
                "first_solution_iteration": stats.get("first_solution_iteration"),
                "number_of_nodes": stats.get("number_of_nodes"),
                "max_transforms": stats.get("max_transforms"),
                "status": status,
                "best_state_score": best_state_score,
            })

    total_time = time.time() - t0
    print(f"Total retrosynthesis time (seconds): {total_time:.2f}")

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    print(f"Saved search stats to {RESULTS_CSV}")
    print(f"Saved all routes to {ROUTES_JSONL}")


# ------------------------------
# MAIN
# ------------------------------
if __name__ == "__main__":
    # 1. Конвертация SDF -> SMILES при необходимости
    if not SMILES_TXT.exists():
        sdf_to_smiles(SDF_INPUT, SMILES_TXT)

    # 2. Единственный прогон ретросинтеза и сохранение всего нужного
    run_retrosynthesis_and_save()


### SynthScore 

In [ ]:
from pathlib import Path

BASE_DIR = Path(".").resolve()

SDF_IN   = BASE_DIR / "output_500.sdf"
CSV_IN   = BASE_DIR / "results_500.csv"
SDF_OUT  = BASE_DIR / "output_500_scored.sdf"

PENALTY_NO_ROUTE = 15.0

In [ ]:
import pandas as pd
from rdkit import Chem

df_res = pd.read_csv(CSV_IN)
df_res.columns = [c.strip() for c in df_res.columns]

required = [
    "target", "n_routes", "search_time", "first_solution_time",
    "first_solution_iteration", "number_of_nodes", "max_transforms",
    "status", "best_state_score"
]
missing = [c for c in required if c not in df_res.columns]
if missing:
    raise ValueError(f"results_500.csv missing columns: {missing}")

def canon_smiles(smi: str):
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True) if m else None

# Канонизируем target и делаем быстрый маппинг SMILES -> строка
df_res["target_canon"] = df_res["target"].map(canon_smiles)
if df_res["target_canon"].isna().any():
    bad = df_res[df_res["target_canon"].isna()].head(5)["target"].tolist()
    raise ValueError(f"Some targets cannot be parsed as SMILES, examples: {bad}")

# На всякий случай контроль уникальности
if df_res["target_canon"].duplicated().any():
    dups = df_res[df_res["target_canon"].duplicated(keep=False)].head(10)
    raise ValueError("Duplicate canonical targets detected:\n" + str(dups[["target","target_canon"]]))

res_by_smi = df_res.set_index("target_canon").to_dict(orient="index")

print("Loaded:", len(df_res), "rows")
print("Status counts:\n", df_res["status"].value_counts(dropna=False))
print("n_routes min/max:", df_res["n_routes"].min(), df_res["n_routes"].max())

In [ ]:
import math, time
from rdkit import Chem
from rdkit.Contrib.SA_Score import sascorer  # SA_score

def synthscore_from_row(row, sa_score: float, penalty=15.0):
    status = str(row["status"])
    n_routes = int(row["n_routes"]) if pd.notna(row["n_routes"]) else 0
    success = (status == "solved") and (n_routes > 0)

    if not success:
        return penalty + sa_score, "no_route"

    max_transforms   = int(row["max_transforms"]) if pd.notna(row["max_transforms"]) else 0
    best_state_score = float(row["best_state_score"]) if pd.notna(row["best_state_score"]) else 0.0
    number_of_nodes  = int(row["number_of_nodes"]) if pd.notna(row["number_of_nodes"]) else 0
    search_time      = float(row["search_time"]) if pd.notna(row["search_time"]) else 0.0

    score = (
        1.0 * max_transforms +
        3.0 * (1.0 - best_state_score) +
        0.5 * math.log10(1.0 + number_of_nodes) +
        0.5 * (search_time / 60.0)
    )
    return float(score), "found"


# --- измеряем ТОЛЬКО расчёт скоринга (без записи в файл) ---

suppl = Chem.SDMolSupplier(str(SDF_IN), removeHs=False)

# (Опционально) прогрев на нескольких молекулах, чтобы исключить разовый overhead
for i, mol in enumerate(suppl):
    if mol is None:
        continue
    _ = float(sascorer.calculateScore(Chem.RemoveHs(mol)))
    if i >= 4:
        break

# пересоздаём supplier заново для честного измерения по всем 500
suppl = Chem.SDMolSupplier(str(SDF_IN), removeHs=False)

t0 = time.perf_counter()

n_total = n_matched = n_missing = 0
n_found_case = n_no_route_case = 0
min_score = float("inf")
max_score = float("-inf")

for mol in suppl:
    if mol is None:
        continue
    n_total += 1

    smi = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    smi_canon = canon_smiles(smi)
    row = res_by_smi.get(smi_canon)

    sa = float(sascorer.calculateScore(Chem.RemoveHs(mol)))

    if row is None:
        n_missing += 1
        score, case = PENALTY_NO_ROUTE + sa, "missing"
    else:
        n_matched += 1
        score, case = synthscore_from_row(row, sa, penalty=PENALTY_NO_ROUTE)

    if case == "found":
        n_found_case += 1
    elif case == "no_route":
        n_no_route_case += 1

    min_score = min(min_score, score)
    max_score = max(max_score, score)

t1 = time.perf_counter()

print("=== SCORING (NO WRITE) DONE ===")
print("Total mols:", n_total)
print("Matched to results_500.csv:", n_matched)
print("Missing in results_500.csv:", n_missing)
print("Case found:", n_found_case, "| Case no_route:", n_no_route_case)
print("SynthScore min/max:", min_score, max_score)
print(f"Scoring-only time (sec): {t1 - t0:.2f}")
print(f"Per-molecule (ms): {(t1 - t0) * 1000 / max(1, n_total):.2f}")
